# Ejercicio 02: Model Registry con MLflow

## Objetivo

En este ejercicio vas a aprender a usar el **Model Registry** de MLflow paso a paso.

Ya sabes hacer tracking (parametros, metricas, artefactos). Ahora vas a dar el siguiente paso:
**registrar modelos con nombre y version**, asignarles etiquetas (`champion`, `candidate`),
y cargarlos para hacer predicciones — exactamente como se hace en produccion.

---

### Analogia: La vitrina de la panaderia

Imagina que tienes una panaderia:

| Panaderia | MLflow |
|---|---|
| Cocina (donde experimentas recetas) | **Tracking** (runs, metricas, parametros) |
| Vitrina (donde pones los productos aprobados) | **Model Registry** (modelos registrados) |
| Etiqueta "Producto estrella" en la vitrina | **Alias `champion`** |
| Etiqueta "Nuevo, en evaluacion" | **Alias `candidate`** |
| Receta v1, v2, v3... | **Versiones** del modelo |

El tracking es tu cocina donde experimentas. El Registry es la vitrina donde pones
solo los modelos que ya estan listos.

---

### Que vas a hacer en este ejercicio:

1. Entrenar un modelo de clasificacion (Iris) y loggearlo en MLflow
2. **Registrarlo** en el Model Registry (darle nombre y version)
3. Asignarle el **alias `champion`**
4. **Cargar** el modelo desde el Registry (por version y por alias)
5. Hacer **predicciones** con el modelo cargado
6. Entrenar un segundo modelo, registrarlo como **v2** y asignarle `candidate`

---

### Flujo visual:

```
Tracking (cocina)                    Registry (vitrina)              Produccion
+-----------------------+    +----------------------------+    +----------------+
| run 1: RandomForest   | -> | iris-clasificacion          | -> |                |
| run 2: otro modelo    |    |   v1 (champion)            |    |  .predict()    |
|                       |    |   v2 (candidate)           |    |                |
+-----------------------+    +----------------------------+    +----------------+
```

---

## Prerrequisitos

### 1. Tener las librerias instaladas

In [ ]:
# Descomentar si necesitas instalar
# !pip install mlflow scikit-learn

### 2. Tener el servidor de MLflow corriendo

Abre una **terminal aparte** y ejecuta:

```bash
mlflow server \
  --host 127.0.0.1 \
  --port 5001 \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlruns
```

Verifica que la UI esta disponible en: http://127.0.0.1:5001

> **Importante**: El Model Registry necesita un backend con base de datos (como SQLite).
> Si solo usas `mlflow ui` sin `--backend-store-uri`, el Registry **no funcionara**.

---

## Parte 1: Codigo base (NO modificar, solo ejecutar)

Las siguientes celdas cargan el dataset Iris, entrenan un modelo y calculan metricas.
**Ejecutalas todas** antes de continuar con los TODOs.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# --- Cargar dataset Iris ---
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

print(f"Dataset: {len(X)} registros, {X.shape[1]} features")
print(f"Clases: {list(iris.target_names)}")
print(f"Distribucion: {np.bincount(y)}")

In [ ]:
# --- Dividir datos ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

In [ ]:
# --- Escalar features ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Features escaladas con StandardScaler")

In [ ]:
# --- Entrenar modelo ---
n_estimators = 100
max_depth = 5
random_state = 42

modelo = RandomForestClassifier(
    n_estimators=n_estimators,
    max_depth=max_depth,
    random_state=random_state,
)
modelo.fit(X_train_scaled, y_train)
print("Modelo entrenado!")

In [ ]:
# --- Calcular metricas ---
y_pred = modelo.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print("\nReporte de clasificacion:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

---

## Hasta aqui todo funciona, pero...

El modelo vive solo en la memoria de Python. Si cierras el notebook, **se pierde**.
No hay forma de:
- Encontrarlo por nombre
- Saber que version es
- Cargarlo desde otro script
- Saber cual es el modelo "oficial" en produccion

**Tu tarea:** Registrar este modelo en el Model Registry de MLflow.

---

## Parte 2: Completar los TODOs

### TODO 1: Conectar a MLflow y crear un experimento

Lo primero es conectarte al servidor de MLflow y seleccionar (o crear) un experimento.

**Que debes hacer:**

1. Importar `mlflow`
2. Conectarte al servidor con `mlflow.set_tracking_uri("http://127.0.0.1:5001")`
3. Crear/seleccionar un experimento con `mlflow.set_experiment("iris-model-registry")`

**Resultado esperado:** Un mensaje confirmando la conexion.

In [ ]:
# TODO 1: Conectar a MLflow
# Escribe tu codigo aqui:




print(f"Tracking URI: {mlflow.get_tracking_uri()}")

### TODO 2: Loggear el modelo en un run de MLflow

Antes de registrar un modelo en el Registry, primero necesitas **loggearlo dentro de un run**.
Es como cocinar la receta antes de ponerla en la vitrina.

**Que debes hacer:**

Dentro de `with mlflow.start_run(run_name="rf_iris_v1") as run:` haz lo siguiente:

1. **Registrar parametros** con `mlflow.log_param(nombre, valor)`:
   - `"n_estimators"` -> `n_estimators` (la variable definida arriba, vale 100)
   - `"max_depth"` -> `max_depth` (vale 5)
   - `"random_state"` -> `random_state` (vale 42)

2. **Registrar la metrica** con `mlflow.log_metric(nombre, valor)`:
   - `"accuracy"` -> `accuracy` (la variable calculada arriba)

3. **Loggear el modelo** con `mlflow.sklearn.log_model(modelo, "modelo_rf")`
   - Primer argumento: el objeto del modelo (`modelo`)
   - Segundo argumento: el nombre de la carpeta dentro de artifacts (`"modelo_rf"`)

4. **Guardar el run_id** para usarlo despues: `run_id = run.info.run_id`

**Resultado esperado:** Un enlace al run en la UI de MLflow.

In [ ]:
# TODO 2: Loggear modelo en un run
# Escribe tu codigo aqui:

with mlflow.start_run(run_name="rf_iris_v1") as run:

    # 2a. Registrar parametros (3 log_param)



    # 2b. Registrar metrica (1 log_metric)


    # 2c. Loggear el modelo (1 log_model)


    # 2d. Guardar run_id
    run_id = run.info.run_id


print(f"Run ID: {run_id}")
print(f"Accuracy: {accuracy:.4f}")

### Verificacion del TODO 2

Ejecuta la celda de abajo. Si ves parametros y metricas, el TODO 2 esta bien.

In [ ]:
# Verificacion
run_data = mlflow.get_run(run_id)
print("Parametros registrados:", dict(run_data.data.params))
print("Metricas registradas: ", dict(run_data.data.metrics))
print("Artifacts:")
from mlflow.tracking import MlflowClient
client = MlflowClient()
for artifact in client.list_artifacts(run_id):
    print(f"  -> {artifact.path}")

---

### TODO 3: Registrar el modelo en el Model Registry

Ahora viene lo nuevo. Hasta ahora el modelo esta "loggeado" dentro de un run (esta en la cocina).
Vamos a **registrarlo** en el Registry (ponerlo en la vitrina).

**Concepto clave:**
- Un **modelo registrado** tiene un **nombre** (como `"iris-clasificacion"`) y **versiones** (v1, v2, ...).
- Cada version apunta a un run especifico.
- Puedes tener multiples versiones del mismo modelo.

**Que debes hacer:**

1. Definir el nombre del modelo: `nombre_modelo = "iris-clasificacion"`
2. Construir la URI del modelo loggeado: `model_uri = f"runs:/{run_id}/modelo_rf"`
   - `runs:/` indica que viene de un run
   - `{run_id}` es el ID del run donde esta
   - `/modelo_rf` es el nombre que usaste en `log_model`
3. Registrar con: `mlflow.register_model(model_uri, nombre_modelo)`

**Resultado esperado:** Un mensaje indicando que se creo la version 1 del modelo.

In [ ]:
# TODO 3: Registrar modelo en el Registry
# Escribe tu codigo aqui:

# 3a. Definir nombre del modelo


# 3b. Construir model_uri (ruta al modelo dentro del run)


# 3c. Registrar el modelo


print(f"Modelo '{nombre_modelo}' registrado!")

### Que deberias ver en MLflow UI

Abre http://127.0.0.1:5001 y ve a la seccion **Models** (barra lateral izquierda):

```
MLflow UI -> Models
  └── iris-clasificacion
       └── Version 1
            ├── Source Run: rf_iris_v1
            └── Aliases: (ninguno aun)
```

Si ves eso, el TODO 3 esta correcto.

---

### TODO 4: Asignar el alias `champion` a la version 1

Los **aliases** son etiquetas que puedes mover entre versiones. Son la forma moderna
(MLflow 2.x) de indicar que version es la "buena":

- `champion`: la version estable, la que esta en produccion
- `candidate`: la version nueva que estas evaluando

Un alias apunta a **una sola version** a la vez. Si mueves `champion` de v1 a v2,
la v1 deja de ser champion automaticamente.

**Que debes hacer:**

1. Ya tienes `client = MlflowClient()` creado arriba
2. Asignar el alias con:
   ```python
   client.set_registered_model_alias(nombre_modelo, "champion", "1")
   ```
   - Primer argumento: nombre del modelo (`nombre_modelo`)
   - Segundo argumento: nombre del alias (`"champion"`)
   - Tercer argumento: numero de version como string (`"1"`)

**Resultado esperado:** La version 1 ahora tiene el alias `champion`.

In [ ]:
# TODO 4: Asignar alias champion
# Escribe tu codigo aqui:



print(f"Alias 'champion' asignado a version 1 de '{nombre_modelo}'")

### Verificacion del TODO 4

Ejecuta la celda de abajo para confirmar que el alias se asigno correctamente.

In [ ]:
# Verificacion
modelo_info = client.get_registered_model(nombre_modelo)
print(f"Modelo: {modelo_info.name}")
print(f"Aliases: {modelo_info.aliases}")

# Verificar que champion apunta a version 1
champion = client.get_model_version_by_alias(nombre_modelo, "champion")
print(f"Champion es la version: {champion.version}")

### Que deberias ver en MLflow UI

```
MLflow UI -> Models -> iris-clasificacion
  └── Version 1
       ├── Aliases: champion     <-- Nuevo!
       └── Source Run: rf_iris_v1
```

---

### TODO 5: Cargar el modelo por numero de version

Ahora que el modelo esta registrado, puedes cargarlo **sin tener el archivo local**.
Solo necesitas el nombre y la version.

Hay **3 formas** de cargar un modelo desde el Registry:

| Forma | URI | Cuando usarla |
|-------|-----|---------------|
| Por version | `models:/nombre/1` | Debugging, rollback |
| Por alias | `models:/nombre@champion` | **Produccion** (recomendada) |
| Por stage | `models:/nombre/Staging` | Sistema antiguo (deprecated) |

Empecemos con la forma 1: **por version**.

**Que debes hacer:**

1. Construir la URI: `model_uri_v1 = f"models:/{nombre_modelo}/1"`
2. Cargar el modelo: `modelo_cargado = mlflow.sklearn.load_model(model_uri_v1)`

**Resultado esperado:** El modelo se carga y puedes ver su tipo.

In [ ]:
# TODO 5: Cargar modelo por version
# Escribe tu codigo aqui:

# 5a. Construir la URI (models:/nombre/version)


# 5b. Cargar el modelo


print(f"Modelo cargado: {type(modelo_cargado).__name__}")
print(f"Desde: {model_uri_v1}")

---

### TODO 6: Cargar el modelo por alias (forma recomendada para produccion)

Esta es la **forma mas robusta**. Tu codigo de produccion siempre pide el modelo
`champion`, y cuando quieras actualizar, solo mueves el alias a la nueva version
— **sin cambiar ni una linea de codigo**.

**Que debes hacer:**

1. Obtener la version champion:
   ```python
   champion_info = client.get_model_version_by_alias(nombre_modelo, "champion")
   ```
2. Construir la URI con esa version:
   ```python
   model_uri_champion = f"models:/{nombre_modelo}/{champion_info.version}"
   ```
3. Cargar el modelo:
   ```python
   modelo_champion = mlflow.sklearn.load_model(model_uri_champion)
   ```

**Resultado esperado:** El mismo modelo cargado, pero esta vez via el alias.

In [ ]:
# TODO 6: Cargar modelo por alias
# Escribe tu codigo aqui:

# 6a. Obtener la version del champion


# 6b. Construir la URI


# 6c. Cargar el modelo


print(f"Modelo champion cargado (version {champion_info.version})")
print(f"Desde: {model_uri_champion}")

---

### TODO 7: Hacer predicciones con el modelo cargado

El modelo cargado desde el Registry funciona exactamente igual que el original.
Vamos a verificarlo haciendo predicciones.

**Que debes hacer:**

1. Predecir con el modelo cargado: `y_pred_registry = modelo_champion.predict(X_test_scaled)`
2. Calcular accuracy: `accuracy_registry = accuracy_score(y_test, y_pred_registry)`
3. Comparar con la accuracy original (deberia ser **identica**)

**Resultado esperado:** La accuracy del modelo cargado es la misma que la original.

In [ ]:
# TODO 7: Hacer predicciones con el modelo del Registry
# Escribe tu codigo aqui:

# 7a. Predecir


# 7b. Calcular accuracy


# 7c. Comparar
print(f"Accuracy original:  {accuracy:.4f}")
print(f"Accuracy Registry:  {accuracy_registry:.4f}")
print(f"Son iguales: {accuracy == accuracy_registry}")

---

## Parte 3: Registrar una segunda version (v2)

En la vida real, siempre entrenas modelos nuevos y quieres compararlos con el actual.
Vamos a entrenar un segundo modelo con **diferentes hiperparametros**,
registrarlo como **version 2**, y asignarle el alias `candidate`.

### TODO 8: Entrenar un segundo modelo con hiperparametros diferentes

**Que debes hacer:**

Dentro de `with mlflow.start_run(run_name="rf_iris_v2") as run2:` haz lo siguiente:

1. Definir nuevos hiperparametros:
   - `n_estimators_v2 = 200`
   - `max_depth_v2 = 10`

2. Registrar parametros con `mlflow.log_param()`

3. Crear y entrenar un nuevo RandomForestClassifier con esos parametros

4. Predecir y calcular accuracy

5. Registrar la metrica con `mlflow.log_metric("accuracy", accuracy_v2)`

6. **Loggear Y registrar** el modelo en un solo paso:
   ```python
   mlflow.sklearn.log_model(
       modelo_v2,
       "modelo_rf",
       registered_model_name=nombre_modelo   # <-- Esto lo registra automaticamente!
   )
   ```
   El parametro `registered_model_name` loggea el modelo Y lo registra en el Registry
   en un solo paso. Como el nombre `"iris-clasificacion"` ya existe, crea la **version 2**.

7. Guardar el run_id: `run_id_v2 = run2.info.run_id`

**Resultado esperado:** Version 2 creada en el Registry.

In [ ]:
# TODO 8: Entrenar y registrar un segundo modelo
# Escribe tu codigo aqui:

with mlflow.start_run(run_name="rf_iris_v2") as run2:

    # 8a. Nuevos hiperparametros


    # 8b. Registrar parametros


    # 8c. Crear y entrenar modelo


    # 8d. Predecir y calcular accuracy


    # 8e. Registrar metrica


    # 8f. Loggear Y registrar modelo (con registered_model_name)


    # 8g. Guardar run_id
    run_id_v2 = run2.info.run_id


print(f"Run ID v2: {run_id_v2}")
print(f"Accuracy v2: {accuracy_v2:.4f}")

---

### TODO 9: Asignar alias `candidate` a la version 2

La version 2 es un modelo nuevo que queremos evaluar antes de promoverlo.
Le asignamos el alias `candidate`.

**Que debes hacer:**

Usar `client.set_registered_model_alias(nombre_modelo, "candidate", "2")`

**Resultado esperado:** v1 = champion, v2 = candidate.

In [ ]:
# TODO 9: Asignar alias candidate a version 2
# Escribe tu codigo aqui:



print("Alias 'candidate' asignado a version 2")

### Verificacion final

In [ ]:
# Verificacion: mostrar estado completo del Registry
modelo_info = client.get_registered_model(nombre_modelo)
print(f"Modelo: {modelo_info.name}")
print(f"Aliases: {modelo_info.aliases}")
print()

# Mostrar todas las versiones
for alias_name in ["champion", "candidate"]:
    version_info = client.get_model_version_by_alias(nombre_modelo, alias_name)
    # Buscar la accuracy de ese run
    run_data = mlflow.get_run(version_info.run_id)
    acc = run_data.data.metrics.get("accuracy", "N/A")
    print(f"  {alias_name:10s} -> Version {version_info.version} | Accuracy: {acc}")

### Que deberias ver en MLflow UI

```
MLflow UI -> Models -> iris-clasificacion
  ├── Version 1
  │    ├── Aliases: champion
  │    └── Source Run: rf_iris_v1
  │
  └── Version 2
       ├── Aliases: candidate
       └── Source Run: rf_iris_v2
```

Si ves eso: **ejercicio completado!**

---

## Bonus (opcional): Promover el candidate a champion

Si la version 2 tiene mejor accuracy, puedes mover el alias `champion` a la version 2.
Recuerda: solo mueves el alias, **tu codigo de produccion no cambia**.

```python
# Mover champion de v1 a v2
client.set_registered_model_alias(nombre_modelo, "champion", "2")
```

Pruebalo si quieres y verifica en la UI que el alias se movio.

In [ ]:
# Bonus: Promover candidate a champion (descomentar para ejecutar)
# client.set_registered_model_alias(nombre_modelo, "champion", "2")
# print("Champion actualizado a version 2!")
# print(f"Aliases: {client.get_registered_model(nombre_modelo).aliases}")

---

## Resumen

En este ejercicio aprendiste:

| Concepto | Funcion | Para que sirve |
|---|---|---|
| Loggear modelo | `mlflow.sklearn.log_model()` | Guardar modelo como artifact dentro de un run |
| Registrar modelo | `mlflow.register_model()` | Crear nombre + version en el Registry |
| Loggear + Registrar | `log_model(..., registered_model_name=)` | Hacer ambas cosas en un paso |
| Asignar alias | `client.set_registered_model_alias()` | Etiquetar version como champion/candidate |
| Cargar por version | `models:/nombre/1` | Obtener una version especifica |
| Cargar por alias | `client.get_model_version_by_alias()` | Obtener la version champion |

### Diagrama del flujo completo:

```
Entrenar           Loggear              Registrar            Etiquetar          Cargar
modelo.fit() -->  log_model()  -->  register_model()  -->  set_alias()  -->  load_model()
  (sklearn)        (tracking)          (registry)        (champion/           (produccion)
                                                          candidate)
```

Este es exactamente el flujo que vimos en clase con el notebook `03_mlflow_advanced.ipynb`,
pero simplificado con el dataset Iris.